# Baseline models

This notebook fits and evaluates the benchmark forecasting models on the
cleaned train, validation and test splits.

All baseline models return predictions and ground truth in raw value space.
`ForecastEvaluator` is responsible for transforming predictions into the
required evaluation space and computing the common metrics.

The current evaluation metrics are:

1. **Cumulative log-change MAE**
2. **MASE**
3. **Relative MAE versus Persistence**
4. **Persistence win rate**

The available benchmark models are:

1. **Persistence** — predicts every future horizon using the final target
   value in the context window.
2. **Mean** — predicts every future horizon using the mean target value over
   the context window.
3. **ARIMA** — fits a separate univariate ARIMA model to the one-step log
   changes of each asset and target channel.
4. **VAR** — fits one multivariate VAR model per target channel across all
   assets.
5. **GARCH** — fits a separate GARCH(1,1) model to each asset and target
   channel, with an optional AR(1), constant or zero conditional mean.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Make sure notebook can import from src/
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

from src.data.load_candle_data import load_candle_splits, clean_candle_splits
from src.evaluation.metrics import ForecastEvaluator
from src.models.persistence import PersistenceBaseline
from src.models.mean import MeanBaseline
from src.models.arima import ArimaBaseline
from src.models.var import VarBaseline
from src.models.garch import GarchBaseline
from src.utils.config import load_yaml
from src.utils.metric_tables import make_evaluation_table

Project root: /Users/vishalruparelia/Desktop/Thesis/dynamic_graphs_thesis


In [2]:
DATA_DIR = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "Shared drives/Vishal/data/cached_datasets/"
    "exp-1m-95s-24y/session"
)

CONFIG_PATH = Path("../configs/forecasting.yaml")

## Load the data and clean

In [3]:
config = load_yaml(CONFIG_PATH)

train_raw, val_raw, test_raw = load_candle_splits(DATA_DIR)

train, val, test = clean_candle_splits(
    train_raw,
    val_raw,
    test_raw,
)

print("train samples:", len(train["samples"]))
print("val samples:", len(val["samples"]))
print("test samples:", len(test["samples"]))
print("channels:", test["channels"])
print("assets:", len(test["asset_cols"]))
print("stride:", config['forecasting']['stride'])
print("input features:", config['forecasting']['input_channels'])
print("targets:", config['forecasting']['target_channels'])

train samples: 187
val samples: 42
test samples: 20
channels: ['open', 'high', 'low', 'close', 'volume', 'amount']
assets: 93
stride: 15
input features: ['open', 'high', 'low', 'close', 'volume', 'amount']
targets: ['open', 'high', 'low', 'close']


## Persistence

In [7]:
persistence = PersistenceBaseline.from_config(config)

persistence.fit(
    train_split=train,
    val_split=val,
)

persistence_result = persistence.predict(
    split=test,
    batch_size=256,
)

persistence_evaluator = ForecastEvaluator(
    prediction_result=persistence_result,
    train_split=train,
)

persistence_results = persistence_evaluator.evaluate(
    metrics=persistence_evaluator.available_metrics,
    reduce_dims=(0, 2),
)

persistence_metric_table = make_evaluation_table(
    metric_results=persistence_results,
    horizons=persistence_evaluator.horizons,
    channels=persistence_evaluator.channels,
)

for metric_name in persistence_evaluator.available_metrics:
    metric_pivot = (
        persistence_metric_table
        .loc[
            persistence_metric_table["metric"] == metric_name
        ]
        .pivot(
            index="horizon",
            columns="channel",
            values="value",
        )
    )

    display(
        metric_pivot.style.set_caption(metric_name)
    )

channel,close,high,low,open
horizon,,,,
1,0.000386,0.000338,0.000337,0.000370
5,0.000824,0.000814,0.000819,0.000838
15,0.001364,0.001361,0.001366,0.001376
30,0.001868,0.001871,0.001878,0.001894
60,0.002693,0.002695,0.002711,0.002716


channel,close,high,low,open
horizon,,,,
1,0.982601,0.985617,0.967567,0.961134
5,2.107645,2.378798,2.354249,2.183830
15,3.477875,3.960247,3.916564,3.578878
30,4.754426,5.432416,5.373256,4.913269
60,6.856115,7.833973,7.766035,7.054037


channel,close,high,low,open
horizon,,,,
1,1.000000,1.000000,1.000000,1.000000
5,1.000000,1.000000,1.000000,1.000000
15,1.000000,1.000000,1.000000,1.000000
30,1.000000,1.000000,1.000000,1.000000
60,1.000000,1.000000,1.000000,1.000000


channel,close,high,low,open
horizon,,,,
1,0.500000,0.500000,0.500000,0.500000
5,0.500000,0.500000,0.500000,0.500000
15,0.500000,0.500000,0.500000,0.500000
30,0.500000,0.500000,0.500000,0.500000
60,0.500000,0.500000,0.500000,0.500000


## Mean

In [8]:
mean = MeanBaseline.from_config(config)

mean.fit(
    train_split=train,
    val_split=val,
)

mean_result = mean.predict(
    split=test,
    batch_size=256,
)

mean_evaluator = ForecastEvaluator(
    prediction_result=mean_result,
    train_split=train,
)

mean_results = mean_evaluator.evaluate(
    metrics=mean_evaluator.available_metrics,
    reduce_dims=(0, 2),
)

mean_metric_table = make_evaluation_table(
    metric_results=mean_results,
    horizons=mean_evaluator.horizons,
    channels=mean_evaluator.channels,
)

for metric_name in mean_evaluator.available_metrics:
    metric_pivot = (
        mean_metric_table
        .loc[
            mean_metric_table["metric"] == metric_name
        ]
        .pivot(
            index="horizon",
            columns="channel",
            values="value",
        )
    )

    display(
        metric_pivot.style.set_caption(metric_name)
    )

channel,close,high,low,open
horizon,,,,
1,0.001686,0.001679,0.001689,0.001685
5,0.001845,0.001841,0.001848,0.001850
15,0.002139,0.002138,0.002146,0.002149
30,0.002531,0.002527,0.002539,0.002539
60,0.003207,0.003200,0.003225,0.003218


channel,close,high,low,open
horizon,,,,
1,4.289059,4.873940,4.833848,4.374175
5,4.697048,5.352901,5.296890,4.808750
15,5.435785,6.202680,6.138109,5.573514
30,6.426107,7.327941,7.257336,6.582361
60,8.150916,9.286466,9.225687,8.347090


channel,close,high,low,open
horizon,,,,
1,4.428495,4.819709,4.947495,4.555269
5,2.211558,2.254433,2.237649,2.232673
15,1.571649,1.573500,1.574112,1.565866
30,1.354611,1.348741,1.352954,1.340667
60,1.189438,1.185278,1.188600,1.184427


channel,close,high,low,open
horizon,,,,
1,0.142841,0.126358,0.122977,0.135937
5,0.255122,0.251542,0.252844,0.255291
15,0.337507,0.334918,0.337351,0.336106
30,0.368124,0.366978,0.368110,0.369327
60,0.401203,0.401302,0.401797,0.401613


## ARIMA

In [ ]:
config = load_yaml(CONFIG_PATH)

arima = ArimaBaseline.from_config(
    config,
    fit_mode="auto",
    optim_method="powell",
)

arima.fit(
    train_split=train,
    val_split=val,
)

arima_result = arima.predict(
    split=test,
    batch_size=256,
)

arima_evaluator = ForecastEvaluator(
    prediction_result=arima_result,
    train_split=train,
)

arima_results = arima_evaluator.evaluate(
    metrics=arima_evaluator.available_metrics,
    reduce_dims=(0, 2),
)

arima_metric_table = make_evaluation_table(
    metric_results=arima_results,
    horizons=arima_evaluator.horizons,
    channels=arima_evaluator.channels,
)

for metric_name in arima_evaluator.available_metrics:
    metric_pivot = (
        arima_metric_table
        .loc[
            arima_metric_table["metric"] == metric_name
        ]
        .pivot(
            index="horizon",
            columns="channel",
            values="value",
        )
    )

    display(
        metric_pivot.style.set_caption(metric_name)
    )

In [21]:
order_table = (
    pd.Series(arima_result['selected_orders'].values())
    .value_counts()
    .rename_axis("order")
    .reset_index(name="count")
)

order_table

,order,count
0,"(0, 0, 1)",20
1,"(0, 0, 0)",14
2,"(1, 0, 0)",13
3,"(1, 0, 1)",10
4,"(0, 0, 2)",9
5,"(2, 0, 0)",8
6,"(3, 0, 0)",6
7,"(2, 0, 1)",3
8,"(1, 0, 2)",3
9,"(1, 0, 3)",2


## VAR

In [10]:
var = VarBaseline.from_config(
    config,
    maxlags=15,
    ic="aic",
    trend="c",
)

var.fit(
    train_split=train,
    val_split=val,
)

var_result = var.predict(
    split=test,
    batch_size=256,
)

var_evaluator = ForecastEvaluator(
    prediction_result=var_result,
    train_split=train,
)

var_results = var_evaluator.evaluate(
    metrics=var_evaluator.available_metrics,
    reduce_dims=(0, 2),
)

var_metric_table = make_evaluation_table(
    metric_results=var_results,
    horizons=var_evaluator.horizons,
    channels=var_evaluator.channels,
)

for metric_name in var_evaluator.available_metrics:
    metric_pivot = (
        var_metric_table
        .loc[
            var_metric_table["metric"] == metric_name
        ]
        .pivot(
            index="horizon",
            columns="channel",
            values="value",
        )
    )

    display(
        metric_pivot.style.set_caption(metric_name)
    )

Fitting 4 VAR model(s) with maxlags=15, ic=aic...
  open: selected_lag=11, failed=False
  high: selected_lag=11, failed=False
  low: selected_lag=10, failed=False
  close: selected_lag=11, failed=False
Finished fitting VAR models.
Failed models: 0


channel,close,high,low,open
horizon,,,,
1,0.000397,0.000347,0.000345,0.000379
5,0.000837,0.000825,0.000828,0.000849
15,0.001377,0.001370,0.001376,0.001388
30,0.001877,0.001875,0.001886,0.001899
60,0.002703,0.002692,0.002729,0.002722


channel,close,high,low,open
horizon,,,,
1,1.011666,1.012999,0.991761,0.984322
5,2.140043,2.411740,2.380588,2.212010
15,3.511896,3.988063,3.946256,3.608790
30,4.781707,5.447143,5.403667,4.930155
60,6.886457,7.826226,7.830615,7.071778


channel,close,high,low,open
horizon,,,,
1,1.035947,1.033366,1.024158,1.020947
5,1.021299,1.018305,1.015011,1.018758
15,1.011940,1.009974,1.009085,1.011674
30,1.006149,1.003236,1.005118,1.003893
60,1.004147,1.000821,1.006533,1.003378


channel,close,high,low,open
horizon,,,,
1,0.435526,0.451203,0.450481,0.450325
5,0.472029,0.476769,0.477023,0.476783
15,0.474462,0.485017,0.477165,0.476486
30,0.481239,0.490846,0.480702,0.486220
60,0.481791,0.504598,0.470388,0.489544


## GARCH

In [9]:
garch = GarchBaseline.from_config(
    config,
    mean="AR",
    return_scale=10000.0,
)

garch.fit(
    train_split=train,
    val_split=val,
)

garch_result = garch.predict(
    split=test,
    batch_size=256,
)

garch_evaluator = ForecastEvaluator(
    prediction_result=garch_result,
    train_split=train,
)

garch_results = garch_evaluator.evaluate(
    metrics=garch_evaluator.available_metrics,
    reduce_dims=(0, 2),
)

garch_metric_table = make_evaluation_table(
    metric_results=garch_results,
    horizons=garch_evaluator.horizons,
    channels=garch_evaluator.channels,
)

for metric_name in garch_evaluator.available_metrics:
    metric_pivot = (
        garch_metric_table
        .loc[
            garch_metric_table["metric"] == metric_name
        ]
        .pivot(
            index="horizon",
            columns="channel",
            values="value",
        )
    )

    display(
        metric_pivot.style.set_caption(metric_name)
    )

Fitting 372 GARCH(1,1) models with mean='AR'...
  fitted 25/372
  fitted 50/372
  fitted 75/372
  fitted 100/372
  fitted 125/372
  fitted 150/372
  fitted 175/372
  fitted 200/372
  fitted 225/372
  fitted 250/372
  fitted 275/372
  fitted 300/372
  fitted 325/372
  fitted 350/372
Finished fitting GARCH models.
Failed models: 0


channel,close,high,low,open
horizon,,,,
1,0.000387,0.000339,0.000337,0.000371
5,0.000824,0.000815,0.000819,0.000839
15,0.001365,0.001364,0.001366,0.001377
30,0.001871,0.001878,0.001878,0.001897
60,0.002706,0.002716,0.002714,0.002728


channel,close,high,low,open
horizon,,,,
1,0.984497,0.989159,0.968887,0.962011
5,2.108020,2.381013,2.354376,2.185342
15,3.482428,3.969580,3.918269,3.582694
30,4.765507,5.454404,5.376874,4.922115
60,6.892001,7.901668,7.782429,7.091179


channel,close,high,low,open
horizon,,,,
1,1.004663,1.002416,1.001639,1.000148
5,0.998885,0.999348,0.999165,1.000171
15,1.000819,1.002182,0.999833,1.001388
30,1.001660,1.003362,0.999982,1.001278
60,1.004202,1.009134,1.000545,1.005086


channel,close,high,low,open
horizon,,,,
1,0.466256,0.464021,0.468845,0.456211
5,0.489488,0.486545,0.488483,0.477646
15,0.481975,0.478155,0.490224,0.483560
30,0.478212,0.470826,0.489997,0.480999
60,0.464219,0.454499,0.485031,0.464361
